# 👔 Day 2: Traversal Mechanics & Cycle Protections

Welcome to Day 2. Today, we graduate from a simple counter to processing a live corporate organizational hierarchy. We will map dependencies, track loop metrics, and intentionally build a data loop to test our safeguards.

### Roadmap:
* **1. The Corporate Seed:** Setting up our hierarchical employee dataset.
* **2. The Descendant Walk:** Executing a top-down structural traversal starting from the CEO.
* **3. Middle-Management Pivot:** Shifting the anchor target to isolate sub-trees.
* **4. Cursed Ancestry Loop:** Simulating a circular reference to prove why path arrays are critical.

### Initialize your Spark session (Ensure Spark 4.1.0+ or DBR 17.0+)

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Current Spark Version: 4.1.2


### CASE 1: Building the Corporate Seed
Let's mock up an organizational structure with explicit parent-child manager hierarchies.

In [8]:
print("\n--- Case 1: Seeding hierarchical employee structure ---")

employees_data = [
    ("CEO_001", "Rita Root", None),
    ("VP_001",  "Victor VP", "CEO_001"),
    ("VP_002",  "Valerie VP", "CEO_001"),
    ("DIR_001", "Dina Director", "VP_001"),
    ("MGR_001", "Marco Manager", "DIR_001"),
    ("ENG_001", "Eddie Engineer", "MGR_001"),
    ("ENG_002", "Eva Engineer", "MGR_001")
]

spark.createDataFrame(employees_data, ["employee_id", "employee_name", "manager_id"]) \
     .createOrReplaceTempView("employees")

spark.sql("SELECT * FROM employees ORDER BY employee_id").show(truncate=False)


--- Case 1: Seeding hierarchical employee structure ---
+-----------+--------------+----------+
|employee_id|employee_name |manager_id|
+-----------+--------------+----------+
|CEO_001    |Rita Root     |NULL      |
|DIR_001    |Dina Director |VP_001    |
|ENG_001    |Eddie Engineer|MGR_001   |
|ENG_002    |Eva Engineer  |MGR_001   |
|MGR_001    |Marco Manager |DIR_001   |
|VP_001     |Victor VP     |CEO_001   |
|VP_002     |Valerie VP    |CEO_001   |
+-----------+--------------+----------+



### CASE 2: The Descendant Walk (Top-Down Hierarchy Traversal)
Walking down the organizational chart starting from the top-level root node (`CEO_001`).

In [9]:
print("\n--- Case 2: Running complete descendant traversal from the CEO ---")

spark.sql("""
WITH RECURSIVE reports AS (
  -- Anchor Member: Start tracking at the absolute root node
  SELECT
    employee_id,
    employee_name,
    manager_id,
    0 AS depth,
    array(employee_id) AS path
  FROM employees
  WHERE employee_id = 'CEO_001'

  UNION ALL

  -- Recursive Member: Fetch workers whose manager matches our yesterday frontier
  SELECT
    e.employee_id,
    e.employee_name,
    e.manager_id,
    r.depth + 1 AS depth,
    concat(r.path, array(e.employee_id)) AS path
  FROM employees e
  JOIN reports r ON e.manager_id = r.employee_id
  WHERE r.depth < 20 -- Business Depth Cap
    AND NOT array_contains(r.path, e.employee_id) -- Loop Prevention Wheel
)
SELECT * FROM reports ORDER BY depth, employee_id
""").show(truncate=False)


--- Case 2: Running complete descendant traversal from the CEO ---
+-----------+--------------+----------+-----+--------------------------------------------+
|employee_id|employee_name |manager_id|depth|path                                        |
+-----------+--------------+----------+-----+--------------------------------------------+
|CEO_001    |Rita Root     |NULL      |0    |[CEO_001]                                   |
|VP_001     |Victor VP     |CEO_001   |1    |[CEO_001, VP_001]                           |
|VP_002     |Valerie VP    |CEO_001   |1    |[CEO_001, VP_002]                           |
|DIR_001    |Dina Director |VP_001    |2    |[CEO_001, VP_001, DIR_001]                  |
|MGR_001    |Marco Manager |DIR_001   |3    |[CEO_001, VP_001, DIR_001, MGR_001]         |
|ENG_001    |Eddie Engineer|MGR_001   |4    |[CEO_001, VP_001, DIR_001, MGR_001, ENG_001]|
|ENG_002    |Eva Engineer  |MGR_001   |4    |[CEO_001, VP_001, DIR_001, MGR_001, ENG_002]|
+-----------+---------

### CASE 3: Middle-Management Pivot
Shifting our anchor target down the hierarchy to cleanly isolate local sub-trees.

In [10]:
print("\n--- Case 3: Shifting anchor focus to isolate middle management sub-trees ---")

spark.sql("""
WITH RECURSIVE reports AS (
  -- Anchor Member: Target a local manager instead of the main root
  SELECT employee_id, employee_name, manager_id, 0 AS depth 
  FROM employees 
  WHERE employee_id = 'VP_001'

  UNION ALL

  SELECT
    e.employee_id,
    e.employee_name,
    e.manager_id,
    r.depth + 1 AS depth
  FROM employees e
  JOIN reports r ON e.manager_id = r.employee_id
  WHERE r.depth < 10
)
SELECT * FROM reports ORDER BY depth
""").show(truncate=False)


--- Case 3: Shifting anchor focus to isolate middle management sub-trees ---
+-----------+--------------+----------+-----+
|employee_id|employee_name |manager_id|depth|
+-----------+--------------+----------+-----+
|VP_001     |Victor VP     |CEO_001   |0    |
|DIR_001    |Dina Director |VP_001    |1    |
|MGR_001    |Marco Manager |DIR_001   |2    |
|ENG_001    |Eddie Engineer|MGR_001   |3    |
|ENG_002    |Eva Engineer  |MGR_001   |3    |
+-----------+--------------+----------+-----+



### CASE 4: Simulating a Data Loop (The Punxsutawney HR Trap)
Injecting dirty cyclical relations (`A ➔ B ➔ C ➔ A`) to demonstrate why path conditional loops are vital.

In [11]:
print("\n--- Case 4: Testing cycle isolation against infected data loop structures ---")

bad_org_data = [
    ("A", "Alice", "C"), 
    ("B", "Bob",   "A"), 
    ("C", "Carol", "B")
]
spark.createDataFrame(bad_org_data, ["employee_id", "employee_name", "manager_id"]) \
     .createOrReplaceTempView("employees_bad")

spark.sql("""
WITH RECURSIVE reports AS (
  SELECT
    employee_id,
    employee_name,
    manager_id,
    0 AS depth,
    array(employee_id) AS path
  FROM employees_bad
  WHERE employee_id = 'A'

  UNION ALL

  SELECT
    e.employee_id,
    e.employee_name,
    e.manager_id,
    r.depth + 1,
    concat(r.path, array(e.employee_id))
  FROM employees_bad e
  JOIN reports r ON e.manager_id = r.employee_id
  WHERE r.depth < 50
    AND NOT array_contains(r.path, e.employee_id)
)
SELECT * FROM reports ORDER BY depth
""").show(truncate=False)


--- Case 4: Testing cycle isolation against infected data loop structures ---
+-----------+-------------+----------+-----+---------+
|employee_id|employee_name|manager_id|depth|path     |
+-----------+-------------+----------+-----+---------+
|A          |Alice        |C         |0    |[A]      |
|B          |Bob          |A         |1    |[A, B]   |
|C          |Carol        |B         |2    |[A, B, C]|
+-----------+-------------+----------+-----+---------+



## 📊 Post-Lab Analysis: The Blueprint Lie Unmasked

We graduated to production-ready hierarchy trees, and the outputs are structurally perfect. However, our execution timers just proved that the "3-Minute Shocker" from Day 1 wasn't a fluke. It is a fundamental architectural bottleneck.

### 🛑 The Real Telemetry
* **Case 1 (Seeding Data):** ~21 seconds (Cold-start session overhead).
* **Case 2 (Full CEO Traversal):** **3 minutes 18 seconds** to find 7 people.
* **Case 3 (Middle-Management Pivot):** **2 minutes 34 seconds** to find 5 people.
* **Case 4 (Cursed Data Cycle):** **1 minute 14 seconds** (Terminated early at depth 2 thanks to our path check array!).

> 💡 **Notice the cycle runtime:** Case 4 finished significantly faster because the path array (`[A, B, C]`) instantly choked the thread at iteration 3. Meanwhile, Case 2 had to track branching paths across 5 complete mornings (depth 0 to 4), dragging its feet the entire way.

### 🧠 Why the Groundhog is Moving in Slow Motion
You have just encountered the **Micro-Data Tax** of distributed engines doing iterative loop logic:

1. **The Catalyst Planning Loop:** A recursive CTE isn't planned once. For every single depth level (0 ➔ 4), Spark's execution layer has to pause, evaluate if new rows were produced, compile a brand new physical sub-graph, and schedule new tasks.
2. **Distributed Freight Train Overhead:** Spark is built to haul gigabytes of data across clusters. When you force it to coordinate distributed states and task boundaries for a handful of records across multiple iterative stages, framework orchestration eats 99.9% of your compute time.